# PassME — Custom YOLOv8 Training (Bup Non Tea Bottle Defect)

**Train YOLOv8n on Bup Non Tea bottle damage dataset. Total time on Colab T4 GPU: ~20-30 minutes.**

## How to use
1. Open this notebook in Google Colab (File → Upload notebook)
2. Runtime → Change runtime type → **T4 GPU** (free tier OK)
3. Run cells in order top-to-bottom
4. Paste your Roboflow snippet in Cell 3 when prompted
5. Download `best.pt` at the end → swap into backend

## Cell 1: Install dependencies (~1 min)

In [ ]:
!pip install -q ultralytics==8.3.* roboflow
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

**Expected:** `CUDA available: True` + `Device: Tesla T4` (or similar).
Nếu in `False` → Runtime → Change runtime type → GPU → Save.

## Cell 2: Sanity-check pretrained YOLOv8 (~30 sec)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # auto-downloads ~6 MB
print('Model loaded:', model.task, '|', len(model.names), 'classes pretrained on COCO')

## Cell 3: Download dataset from Roboflow

**PASTE đoạn code anh COPY từ Roboflow Export step bên dưới** (api_key sẽ là chuỗi thật của anh, em để placeholder).

Ví dụ format:
```python
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_KEY_HERE")
project = rf.workspace("your-workspace").project("passme-bottle-defect")
version = project.version(1)
dataset = version.download("yolov8")
```

In [ ]:
# === PASTE ROBOFLOW SNIPPET HERE ===
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_KEY_HERE")
project = rf.workspace("PASTE_YOUR_WORKSPACE").project("passme-bottle-defect")
version = project.version(1)
dataset = version.download("yolov8")

print('Dataset downloaded to:', dataset.location)
!ls {dataset.location}
!cat {dataset.location}/data.yaml

**Expected:** `data.yaml` in ra `names: ['defect']` và `nc: 1`.

## Cell 4: TRAIN (~15-25 min on T4 GPU)

Hyperparameters tuned for ~100 small-dataset images:
- `epochs=100` with `patience=20` early stopping if no improvement
- `batch=16` for T4 (will auto-reduce if OOM)
- `imgsz=640` standard YOLOv8 input
- Heavy augmentation enabled by default in Ultralytics

Watch the mAP@50 climb — aim for >0.5.

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name='passme_bottle',
    plots=True,
    save=True,
    project='runs/detect',
    exist_ok=True,
    device=0,  # GPU 0 = T4
    cache='ram',  # speed up
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    seed=42,
)
print('Training done. Best weights:', results.save_dir)

## Cell 5: Validate on held-out test set

In [ ]:
best_model = YOLO('runs/detect/passme_bottle/weights/best.pt')
metrics = best_model.val(data=f'{dataset.location}/data.yaml', split='test')
print('=== TEST SET METRICS ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

**Đọc kết quả:**
- **mAP@50 ≥ 0.5** → demo OK, deploy được
- **mAP@50 ≥ 0.7** → model rất tốt, demo wow
- **mAP@50 < 0.4** → cảnh báo: thêm data hoặc check label

Báo em số liệu để mình quyết deploy hay tune thêm.

## Cell 6: Quick visual inference test on sample images

In [ ]:
import glob, random
from PIL import Image
import matplotlib.pyplot as plt

# Pick 4 random test images
test_imgs = glob.glob(f'{dataset.location}/test/images/*.jpg')[:4]
if len(test_imgs) < 4:
    test_imgs = glob.glob(f'{dataset.location}/valid/images/*.jpg')[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, p in zip(axes.flat, test_imgs):
    res = best_model(p, conf=0.25, verbose=False)[0]
    annotated = res.plot()
    ax.imshow(annotated[..., ::-1])  # BGR -> RGB
    ax.set_title(f'{p.split("/")[-1]} | {len(res.boxes)} defects')
    ax.axis('off')
plt.tight_layout(); plt.show()

## Cell 7: Download best.pt for backend deployment

In [ ]:
from google.colab import files
import shutil

src = 'runs/detect/passme_bottle/weights/best.pt'
dst = 'bottle_defect.pt'
shutil.copy(src, dst)
print(f'Size: {round(__import__("os").path.getsize(dst)/1024/1024, 2)} MB')
files.download(dst)

**Download xong** → file `bottle_defect.pt` sẽ trong `~/Downloads/` trên Mac anh.

## Bước tiếp: deploy vào backend

```bash
# Trên Mac, terminal
cd "/Users/tommy/Documents/Claude/Projects/Capstone/iOS Demo Plan/backend"
# Backup model crack cũ
cp best.pt best_crack.pt
# Copy model mới
cp ~/Downloads/bottle_defect.pt best.pt
# Restart uvicorn (Ctrl+C ở terminal backend → chạy lại)
```

App.tsx KHÔNG cần sửa — endpoint `/detect` vẫn vậy. Test lại trên iPhone:
- Chụp chai perfect → PASS (xanh)
- Chụp chai damaged → REJECT (đỏ) + box quanh damage